<a href="https://colab.research.google.com/github/Hajer5503/AgriSmart/blob/feature%2Firrigation-rl/modules/irrigation_rl/notebooks/02_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Irrigation RL – Data Preparation

This notebook prepares weather, soil moisture, and crop parameters
for Reinforcement Learning environment design.

Objectives:

- Merge weather and soil moisture datasets
- Aggregate to daily resolution
- Construct crop calendar (Wheat)
- Compute dynamic Kc values
- Compute ETc (crop evapotranspiration)
- Build final RL state dataframe

## 1.Imports

In [30]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from google.colab import files

## 2.Load Data

In [31]:
soil = pd.read_csv(
    "soil_moisture.csv",
    sep=";",
    skiprows=3
)

soil.columns = soil.columns.str.strip()
print("Soil moisture columns:", soil.columns.tolist())
print(f"Soil moisture date range: {soil['time'].min()} to {soil['time'].max()}")

weather = pd.read_csv( "weather.csv", sep=";", skiprows=3)
weather.columns = weather.columns.str.strip()

print("\nWeather columns:", weather.columns.tolist())
print(f"Weather date range: {weather['time'].min()} to {weather['time'].max()}")

Soil moisture columns: ['time', 'soil_moisture_3_to_9cm (m³/m³)', 'soil_moisture_0_to_1cm (m³/m³)', 'soil_moisture_1_to_3cm (m³/m³)', 'soil_moisture_9_to_27cm (m³/m³)', 'soil_moisture_27_to_81cm (m³/m³)']
Soil moisture date range: 2022-11-16T08:00 to 2026-02-14T23:00

Weather columns: ['time', 'temperature_2m (°C)', 'relative_humidity_2m (%)', 'precipitation (mm)', 'et0_fao_evapotranspiration (mm)', 'wind_speed_10m (km/h)']
Weather date range: 2021-03-23T00:00 to 2026-02-15T23:00


In [25]:
soil.tail()

,time,soil_moisture_3_to_9cm (m³/m³),soil_moisture_0_to_1cm (m³/m³),soil_moisture_1_to_3cm (m³/m³),soil_moisture_9_to_27cm (m³/m³),soil_moisture_27_to_81cm (m³/m³)
28475,2026-02-14 19:00:00,0.212,0.201,0.204,0.228,0.242
28476,2026-02-14 20:00:00,0.211,0.201,0.204,0.228,0.242
28477,2026-02-14 21:00:00,0.211,0.201,0.204,0.228,0.242
28478,2026-02-14 22:00:00,0.211,0.201,0.204,0.227,0.242
28479,2026-02-14 23:00:00,0.211,0.205,0.205,0.227,0.242


## 3. Convert Time to Datetime and Filter to Common Range

In [32]:
# Convert time columns to datetime
soil['time'] = pd.to_datetime(soil['time'])
weather['time'] = pd.to_datetime(weather['time'])

# Find common date range
start_date = max(soil['time'].min(), weather['time'].min())
end_date = min(soil['time'].max(), weather['time'].max())

print(f"Common date range: {start_date} to {end_date}")

# Filter both datasets to common range
soil = soil[(soil['time'] >= start_date) & (soil['time'] <= end_date)].reset_index(drop=True)
weather = weather[(weather['time'] >= start_date) & (weather['time'] <= end_date)].reset_index(drop=True)

print(f"\nFiltered soil moisture: {len(soil)} records")
print(f"Filtered weather: {len(weather)} records")

Common date range: 2022-11-16 08:00:00 to 2026-02-14 23:00:00

Filtered soil moisture: 28480 records
Filtered weather: 28480 records


## 4. Merge and Aggregate to Daily

In [33]:
# Merge on datetime
merged = pd.merge(weather, soil, on='time', how='inner')

# Extract date (remove time component)
merged['date'] = merged['time'].dt.date

# Aggregate to daily (mean for continuous vars, sum for precipitation)
daily_agg = merged.groupby('date').agg({
    'temperature_2m (°C)': 'mean',
    'relative_humidity_2m (%)': 'mean',
    'precipitation (mm)': 'sum',
    'et0_fao_evapotranspiration (mm)': 'sum',
    'wind_speed_10m (km/h)': 'mean',
    'soil_moisture_9_to_27cm (m³/m³)': 'mean',
    'soil_moisture_27_to_81cm (m³/m³)': 'mean',
}).reset_index()

daily_agg['date'] = pd.to_datetime(daily_agg['date'])
daily_agg = daily_agg.sort_values('date').reset_index(drop=True)

print(f"Daily aggregated data: {len(daily_agg)} days")
print(f"Date range: {daily_agg['date'].min()} to {daily_agg['date'].max()}")
daily_agg.head()

Daily aggregated data: 1187 days
Date range: 2022-11-16 00:00:00 to 2026-02-14 00:00:00


,date,temperature_2m (°C),relative_humidity_2m (%),precipitation (mm),et0_fao_evapotranspiration (mm),wind_speed_10m (km/h),soil_moisture_9_to_27cm (m³/m³),soil_moisture_27_to_81cm (m³/m³)
0,2022-11-16,21.818750,39.500000,0.3,2.53,4.556250,0.178750,0.230000
1,2022-11-17,20.404167,56.666667,0.0,2.23,5.979167,0.178000,0.229083
2,2022-11-18,20.700000,54.541667,0.0,3.23,8.512500,0.176875,0.228000
3,2022-11-19,18.962500,52.125000,0.0,3.24,10.700000,0.176333,0.227458
4,2022-11-20,14.800000,55.958333,1.4,3.01,16.637500,0.176000,0.227042


## 5. Define Multi-Year Crop Calendar (Wheat)

Wheat cycle: ~175-180 days typically
- Sowing: Early November
- Heading: Mid-April
- Harvest: Mid-June

In [34]:
# Define crop calendar for multiple years
# Wheat is a winter crop: sown in November, harvested in June

crop_seasons = [
    {'year': 2022, 'sowing': '2022-11-01', 'heading': '2023-04-15', 'harvest': '2023-06-15'},
    {'year': 2023, 'sowing': '2023-11-01', 'heading': '2024-04-15', 'harvest': '2024-06-15'},
    {'year': 2024, 'sowing': '2024-11-01', 'heading': '2025-04-15', 'harvest': '2025-06-15'},
    {'year': 2025, 'sowing': '2025-11-01', 'heading': '2026-04-15', 'harvest': '2026-06-15'},
]

# Convert strings to datetime
for season in crop_seasons:
    season['sowing'] = pd.to_datetime(season['sowing'])
    season['heading'] = pd.to_datetime(season['heading'])
    season['harvest'] = pd.to_datetime(season['harvest'])

# Create list of crop active dates
crop_active_dates = []
for season in crop_seasons:
    date_range = pd.date_range(start=season['sowing'], end=season['harvest'], freq='D')
    crop_active_dates.extend(date_range)

crop_active_df = pd.DataFrame({'date': crop_active_dates})
print(f"Total crop active days: {len(crop_active_df)}")
print(f"Crop calendar range: {crop_active_df['date'].min()} to {crop_active_df['date'].max()}")

Total crop active days: 909
Crop calendar range: 2022-11-01 00:00:00 to 2026-06-15 00:00:00


## 6. Filter Data to Crop Active Periods

In [35]:
# Merge to keep only crop active days
rl_df = pd.merge(crop_active_df, daily_agg, on='date', how='inner')

print(f"After filtering to crop active periods: {len(rl_df)} days")
print(f"Date range: {rl_df['date'].min()} to {rl_df['date'].max()}")
print(f"\nYears covered: {sorted(rl_df['date'].dt.year.unique())}")

After filtering to crop active periods: 773 days
Date range: 2022-11-16 00:00:00 to 2026-02-14 00:00:00

Years covered: [np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]


## 7. Calculate Day of Season

In [36]:
def get_day_of_season(date):
    """Calculate day of season for a given date"""
    date = pd.to_datetime(date)
    year = date.year

    # Find which season this date belongs to
    for season in crop_seasons:
        if season['sowing'] <= date <= season['harvest']:
            days_since_sowing = (date - season['sowing']).days
            return days_since_sowing + 1  # Day 1 is sowing day

    return np.nan

rl_df['day_of_season'] = rl_df['date'].apply(get_day_of_season).astype(int)

print("Day of season calculated")
print(f"Day of season range: {rl_df['day_of_season'].min()} to {rl_df['day_of_season'].max()}")

Day of season calculated
Day of season range: 1 to 228


## 8. Define Crop Growth Stages and Calculate Kc

FAO-56 approach for Wheat Kc values across growth stages

In [37]:
# Wheat crop parameters (field-derived data)
wheat_params = {
    'planting_date': '11-15',
    'harvest_date': '05-15',
    'total_duration_days': 150,
    'kc_initial': 0.7,
    'kc_mid': 1.15,
    'kc_end': 0.35,
    'max_crop_height_m': 1.0,
    'root_depth_m': 1.5
}

print("=== Wheat Crop Parameters ===")
print(f"Planting: {wheat_params['planting_date']}")
print(f"Harvest: {wheat_params['harvest_date']}")
print(f"Total Duration: {wheat_params['total_duration_days']} days")
print(f"Root Depth: {wheat_params['root_depth_m']} m")
print(f"Max Height: {wheat_params['max_crop_height_m']} m")
print(f"\nCrop Coefficients (Kc):")
print(f"  Initial (Kc_i): {wheat_params['kc_initial']}")
print(f"  Mid-season (Kc_mid): {wheat_params['kc_mid']}")
print(f"  End of Season (Kc_end): {wheat_params['kc_end']}")

=== Wheat Crop Parameters ===
Planting: 11-15
Harvest: 05-15
Total Duration: 150 days
Root Depth: 1.5 m
Max Height: 1.0 m

Crop Coefficients (Kc):
  Initial (Kc_i): 0.7
  Mid-season (Kc_mid): 1.15
  End of Season (Kc_end): 0.35


In [38]:
# Growth stages (as day of season)
# Using wheat-specific growth stages with field-derived Kc values
stages = {
    'initial': {'start': 1, 'end': 20, 'kc_start': wheat_params['kc_initial'], 'kc_end': wheat_params['kc_initial']},
    'development': {'start': 20, 'end': 50, 'kc_start': wheat_params['kc_initial'], 'kc_end': wheat_params['kc_mid']},
    'mid_season': {'start': 50, 'end': 100, 'kc_start': wheat_params['kc_mid'], 'kc_end': wheat_params['kc_mid']},
    'late_season': {'start': 100, 'end': 150, 'kc_start': wheat_params['kc_mid'], 'kc_end': wheat_params['kc_end']}
}

def calculate_kc(day_of_season):
    """Calculate crop coefficient based on growth stage with linear interpolation"""
    for stage_name, stage in stages.items():
        if stage['start'] <= day_of_season <= stage['end']:
            # Linear interpolation within stage
            stage_length = stage['end'] - stage['start']
            days_in_stage = day_of_season - stage['start']
            progress = days_in_stage / stage_length if stage_length > 0 else 0
            kc = stage['kc_start'] + (stage['kc_end'] - stage['kc_start']) * progress
            return kc

    return np.nan

rl_df['kc'] = rl_df['day_of_season'].apply(calculate_kc)

print("\n=== Growth Stages Summary ===")
for stage_name, stage in stages.items():
    print(f"{stage_name.capitalize():15} (days {stage['start']:3d}-{stage['end']:3d}): Kc {stage['kc_start']:.2f} → {stage['kc_end']:.2f}")

print(f"\nKc values calculated for dataset")
print(f"Kc range: {rl_df['kc'].min():.3f} to {rl_df['kc'].max():.3f}")


=== Growth Stages Summary ===
Initial         (days   1- 20): Kc 0.70 → 0.70
Development     (days  20- 50): Kc 0.70 → 1.15
Mid_season      (days  50-100): Kc 1.15 → 1.15
Late_season     (days 100-150): Kc 1.15 → 0.35

Kc values calculated for dataset
Kc range: 0.350 to 1.150


## 9. Calculate ETc (Crop Evapotranspiration)

In [39]:
# ETc = Kc * ET0
rl_df['etc'] = rl_df['kc'] * rl_df['et0_fao_evapotranspiration (mm)']

print("ETc calculated")
print(f"ETc range: {rl_df['etc'].min():.3f} to {rl_df['etc'].max():.3f} mm")

ETc calculated
ETc range: 0.420 to 5.727 mm


## 10. Prepare Final RL Dataset

In [40]:
# Select columns for RL state
rl_df_final = rl_df[[
    'date',
    'day_of_season',
    'precipitation (mm)',
    'et0_fao_evapotranspiration (mm)',
    'temperature_2m (°C)',
    'relative_humidity_2m (%)',
    'wind_speed_10m (km/h)',
    'kc',
    'etc',
    'soil_moisture_9_to_27cm (m³/m³)',
    'soil_moisture_27_to_81cm (m³/m³)'
]].copy()

# Round for cleaner output
numeric_cols = rl_df_final.select_dtypes(include=['float64']).columns
rl_df_final[numeric_cols] = rl_df_final[numeric_cols].round(4)

print(f"Final RL dataset: {len(rl_df_final)} records")
print(f"Date range: {rl_df_final['date'].min()} to {rl_df_final['date'].max()}")
print(f"\nYears covered: {sorted(rl_df_final['date'].dt.year.unique())}")
print(f"\nDataset shape: {rl_df_final.shape}")
print(f"\nColumns: {rl_df_final.columns.tolist()}")
rl_df_final.head(10)

Final RL dataset: 773 records
Date range: 2022-11-16 00:00:00 to 2026-02-14 00:00:00

Years covered: [np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

Dataset shape: (773, 11)

Columns: ['date', 'day_of_season', 'precipitation (mm)', 'et0_fao_evapotranspiration (mm)', 'temperature_2m (°C)', 'relative_humidity_2m (%)', 'wind_speed_10m (km/h)', 'kc', 'etc', 'soil_moisture_9_to_27cm (m³/m³)', 'soil_moisture_27_to_81cm (m³/m³)']


,date,day_of_season,precipitation (mm),et0_fao_evapotranspiration (mm),temperature_2m (°C),relative_humidity_2m (%),wind_speed_10m (km/h),kc,etc,soil_moisture_9_to_27cm (m³/m³),soil_moisture_27_to_81cm (m³/m³)
0,2022-11-16,16,0.3,2.53,21.8188,39.5000,4.5562,0.700,1.7710,0.1788,0.2300
1,2022-11-17,17,0.0,2.23,20.4042,56.6667,5.9792,0.700,1.5610,0.1780,0.2291
2,2022-11-18,18,0.0,3.23,20.7000,54.5417,8.5125,0.700,2.2610,0.1769,0.2280
3,2022-11-19,19,0.0,3.24,18.9625,52.1250,10.7000,0.700,2.2680,0.1763,0.2275
4,2022-11-20,20,1.4,3.01,14.8000,55.9583,16.6375,0.700,2.1070,0.1760,0.2270
5,2022-11-21,21,0.0,3.63,15.2708,49.3333,15.6625,0.715,2.5954,0.1760,0.2270
6,2022-11-22,22,0.0,4.01,16.1542,39.5833,19.6250,0.730,2.9273,0.1755,0.2259
7,2022-11-23,23,0.0,3.34,17.7375,46.2083,12.6542,0.745,2.4883,0.1715,0.2213
8,2022-11-24,24,0.0,2.41,18.9958,63.1250,6.0792,0.760,1.8316,0.1694,0.2190
9,2022-11-25,25,0.0,2.29,18.2375,70.3750,6.3292,0.775,1.7748,0.1690,0.2182


## 11. Data Quality Check

In [41]:
print("=== Data Quality Summary ===")
print(f"\nTotal records: {len(rl_df_final)}")
print(f"Date range: {rl_df_final['date'].min()} to {rl_df_final['date'].max()}")
print(f"\nMissing values:")
print(rl_df_final.isnull().sum())

print(f"\nRecords per year:")
print(rl_df_final['date'].dt.year.value_counts().sort_index())

print(f"\nBasic statistics:")
print(rl_df_final.describe())

=== Data Quality Summary ===

Total records: 773
Date range: 2022-11-16 00:00:00 to 2026-02-14 00:00:00

Missing values:
date                                  0
day_of_season                         0
precipitation (mm)                    0
et0_fao_evapotranspiration (mm)       0
temperature_2m (°C)                   0
relative_humidity_2m (%)              0
wind_speed_10m (km/h)                 0
kc                                  232
etc                                 232
soil_moisture_9_to_27cm (m³/m³)       0
soil_moisture_27_to_81cm (m³/m³)      0
dtype: int64

Records per year:
date
2022     46
2023    227
2024    228
2025    227
2026     45
Name: count, dtype: int64

Basic statistics:
                                date  day_of_season  precipitation (mm)  \
count                            773     773.000000          773.000000   
mean   2024-06-02 12:36:19.560155392     107.908150            0.826132   
min              2022-11-16 00:00:00       1.000000            0.000000 

## 12. Save Prepared Dataset

In [42]:
rl_df_final.to_csv("prepared_rl_dataset_extended.csv", index=False)
print("Dataset saved as 'prepared_rl_dataset_extended.csv'")
print(f"\nFile contains {len(rl_df_final)} records spanning {rl_df_final['date'].dt.year.nunique()} years")

Dataset saved as 'prepared_rl_dataset_extended.csv'

File contains 773 records spanning 5 years


In [43]:
files.download("prepared_rl_dataset_extended.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>